In [2]:
import numpy as np 
import math
from scipy.special import jv
import matplotlib.pyplot as plt

In [3]:
f1 = 50 
f2 = 20000

w1 = 2*math.pi*f1
w2 = 2*math.pi*f2

dp_lf = 0.3866424e-3
ds_lf = 1.5235043e-3

con_copper =  59.6e6 #conductivity of copper
my_0 = 4*math.pi*1e-7


skin_depth_1 = np.sqrt(2/(w1*1*my_0*con_copper))
skin_depth_2 = np.sqrt(2/(w2*1*my_0*con_copper))
print(skin_depth_1, skin_depth_2)

xip_lf = dp_lf/(np.sqrt(2)*skin_depth_1)
xis_lf = ds_lf/(np.sqrt(2)*skin_depth_1)

0.009219598309500171 0.0004609799154750085


In [4]:
def calc_N_windings(Voltage, B, w, A_core):
    """
    Number of Windings

    Parameters
    ----------
    Voltage : float or ndarray
        voltage of the coil

    B :  float or ndarray
        selected magnetic flux density in T

    w : float or ndarray
        omega, 2*pi*frequency

    A_core: float or ndarray
        cross-sectional area of the core

    Returns
    -------
    N_w: number of Windings with these parameters
    """
    N_w = Voltage/(w*B*A_core)
    return N_w



In [5]:
"""Low-frequency"""
N_p_lf = calc_N_windings(230*np.sqrt(2), 1.6, w1, 20e-3*27.4e-3)     #number of primary side windings lf-Transformer
N_s_lf = calc_N_windings( 15*np.sqrt(2), 1.6, w1, 20e-3*27.4e-3)     #number of secondary side windings lf-Transformer

"""High-frequency B_VALUE JUST SELECTED WITH 30s OF RESEARCH, INVESTIGATE FURTHER (in this case i selected the higest line of the N27 Core loss graphic)"""
N_p_hf = calc_N_windings(230*np.sqrt(2), 0.3, w2, 20e-3*27.4e-3)     #number of primary side windings lf-Transformer
N_s_hf = calc_N_windings( 15*np.sqrt(2), 0.3, w2, 20e-3*27.4e-3)     #number of secondary side windings lf-Transformer

print(N_p_lf,N_s_lf, N_p_hf, N_s_hf)

1180.8437084633576 77.01154620413203 15.744582779511438 1.026820616055094


In [6]:
"""LF TRANSFORMER MATRIX CALCULATIONS"""

# Known values
V1 = 230 + 0j
I1 = 0.587 + 0j

# Complex impedances
L1 = 1j * w1 * 146.50825
L2 = 1j * w1 * 0.60775797
M  = 1j * w1 * 9.435982714

print(L1, L2, M)

# Solve for unknown I2
I2 = (V1 - L1 * I1) / M
print((M * I1) , (L2 * I2))

# Calculate unknown V2
V2 = (M * I1) + (L2 * I2)

# Results
print(f"I2 = {I2:.6f} A")
print(f"V2 = {V2:.6f} V")

print(f"|I2| = {np.abs(I2):.6f} A")
print(f"|V2| = {np.abs(V2):.6f} V")

print(f"I2 phase = {np.angle(I2, deg=True):.2f}°")
print(f"V2 phase = {np.angle(V2, deg=True):.2f}°")

print("P1 = ", np.real(V1*I1))
print("P2 = ", np.real(V2*I2))
print("Efficency: ", 100*np.real(V2*I2)/np.real(V1*I1))


46026.92418902968j 190.9327973712646j 2964.4013973702677j
1740.103620256347j (14.813966635674786-1740.17762789906j)
I2 = -9.114084-0.077587j A
V2 = 14.813967-0.074008j V
|I2| = 9.114415 A
|V2| = 14.814151 V
I2 phase = -179.51°
V2 phase = -0.29°
P1 =  135.01
P2 =  -135.02148411132117
Efficency:  -100.0085061190439


# Losses

## skineffect + DC resistance
all kinds of support formulas

In [7]:
def ber(n, xi):
    """Kelvin function ber of order n."""
    z = xi * np.exp(3j * np.pi / 4)
    return np.real(jv(n, z))


def bei(n, xi):
    """Kelvin function bei of order n."""
    z = xi * np.exp(3j * np.pi / 4)
    return np.imag(jv(n, z))


def G_R(xi, d):
    """
    Proximity-effect coefficient for a round conductor.

    Parameters
    ----------
    xi : float or ndarray
        Dimensionless parameter d / (sqrt(2) * delta).
    d : float
        Conductor diameter [m].

    Returns
    -------
    G_R : float or ndarray
        Proximity-effect coefficient [m^2].
    """

    ber0 = ber(0, xi)
    ber1 = ber(1, xi)
    ber2 = ber(2, xi)

    bei0 = bei(0, xi)
    bei1 = bei(1, xi)
    bei2 = bei(2, xi)

    denominator = ber0**2 + bei0**2

    term1 = (ber2 * ber1 + ber2 * bei1) / denominator

    term2 = (bei2 * bei1 - bei2 * ber1) / denominator

    GR = -(xi * np.pi**2 * d**2) / (2 * np.sqrt(2)) * (term1 + term2)

    return GR

def F_R(xi):
    """
    Skin-effect resistance factor for a round conductor.

    Parameters
    ----------
    xi : float or ndarray
        Dimensionless parameter d / (sqrt(2) * delta).

    Returns
    -------
    F_R : float or ndarray
        Skin-effect resistance factor (dimensionless).
    """

    ber0 = ber(0, xi)
    ber1 = ber(1, xi)

    bei0 = bei(0, xi)
    bei1 = bei(1, xi)

    denominator = ber1**2 + bei1**2

    term1 = (ber0 * bei1 - ber0 * ber1) / denominator

    term2 = (bei0 * ber1 + bei0 * bei1) / denominator

    FR = (xi / (4 * np.sqrt(2))) * (term1 - term2)

    return FR

def calculate_L(x):
    #calculate length of one turn in mm
    L = 2*(20+27.4)+8*(x-10)
    return L
def calculate_L_coil(x,N):
    #calculate length of whole wire in m
    L = calculate_L(x)*N/1000
    return L

Formulas for DC and Skin resistance

In [8]:
def r_dc(d_i, sigma=con_copper):
    """
    DC resistance per unit length of one strand [Ohm/m].
    """
    return 4/(sigma * np.pi * d_i * d_i)

def P_dc_skin(xi, d_i, n_str, I_hat, L_coil):
    """
    DC + skin-effect losses of Litz wire [W].

    xi     : Dimensionless parameter
    d_i    : Individual strand diameter [m]
    n_str  : Number of strands
    I_hat  : Peak current through the complete Litz wire [A]
    L_coil : Total conductor length [m]
    """

    R = r_dc(d_i)

    P = n_str * R * F_R(xi) * (I_hat / n_str)**2 * L_coil

    return P


In [9]:
P_p_lf = P_dc_skin(xip_lf, dp_lf, 1, I1*np.sqrt(2),calculate_L_coil(22.1,N_p_lf))
P_s_lf = P_dc_skin(xis_lf, ds_lf, 1, I2*np.sqrt(2),calculate_L_coil(22.1,N_s_lf))

print("Powerloss primary: ",P_p_lf)
print("Powerloss secondary: ",P_s_lf)
"""
print("diameter: ", dp_lf)
print("Xi: ",xip_lf)
print("L_coil: ", calculate_L_coil(22.1,N_p_lf))
print("I: ", I1*np.sqrt(2))
print("R_dc:", r_dc(dp_lf, sigma=con_copper)*calculate_L_coil(22.1,N_p_lf))

print(5*math.pi*(dp_lf*1000/2)**2)
"""

Powerloss primary:  (11.140611562571712+0j)
Powerloss secondary:  (11.280357830971266+0.1920711459596156j)


'\nprint("diameter: ", dp_lf)\nprint("Xi: ",xip_lf)\nprint("L_coil: ", calculate_L_coil(22.1,N_p_lf))\nprint("I: ", I1*np.sqrt(2))\nprint("R_dc:", r_dc(dp_lf, sigma=con_copper)*calculate_L_coil(22.1,N_p_lf))\n\nprint(5*math.pi*(dp_lf*1000/2)**2)\n'

Formula for internal proximity:

In [10]:
def P_internal_proximity(xi, d_i, d_a, n_str, I_hat, L_coil):
    """
    Internal proximity-effect losses of Litz wire [W].

    xi     : Dimensionless parameter
    d_i    : Individual strand diameter [m]
    d_a    : Outer diameter of complete Litz wire [m]
    n_str  : Number of strands
    I_hat  : Peak current through the complete Litz wire [A]
    L_coil : Total conductor length [m]
    """

    R = r_dc(d_i)

    P = (n_str * R * G_R(xi, d_i)
         * I_hat**2 / (2 * np.pi**2 * d_a**2)
         * L_coil)

    return P

In [14]:
# Calculate internal proximity losses
P_prox_p_lf = P_internal_proximity(xip_lf, dp_lf, dp_lf, 1, np.abs(I1)*np.sqrt(2), calculate_L_coil(22.1,N_p_lf))
P_prox_s_lf = P_internal_proximity(xip_lf, ds_lf, ds_lf, 1, np.abs(I2)*np.sqrt(2), calculate_L_coil(22.1,N_s_lf))

print("Internal Proximity Primary: ",P_prox_p_lf)
print("Internal Proximity Secondary: ",P_prox_s_lf)

Internal Proximity Primary:  2.692086215714535e-07
Internal Proximity Secondary:  2.726247843279373e-07
